# Task 4: Trend Prediction Analysis (release-year forecasting)


**Goal:** look at how many titles in the Netflix catalog were released each year, build a few simple forecasting models, test them honestly, and forecast the next few years.

**What this is, and is not:** it forecasts the *count of titles in this catalog sample, by release year*. It is not a forecast of what Netflix will commission or what viewers will want. The data also stops on **Sep 25, 2021** (the last `date_added`), so the 2021 release year is incomplete.

Steps (from the task guidelines): 1) prepare release-year data, 2) analyze yearly trends, 3) build forecasting models, 4) visualize predictions, 5) interpret results.

In [1]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs('charts', exist_ok=True)
path = 'netflix_clean.csv' if os.path.exists('netflix_clean.csv') else '../netflix_clean.csv'
df = pd.read_csv(path)
print(df.shape)
df[['type','release_year','date_added']].head()

(8787, 20)


,type,release_year,date_added
0,Movie,2020,2021-09-25
1,TV Show,2021,2021-09-24
2,TV Show,2021,2021-09-24
3,Movie,2021,2021-09-22
4,Movie,1993,2021-09-24


## Step 1: Prepare release-year data
We count titles per `release_year` (not `date_added`, because the task is about release-year trends). Very old years have only a handful of titles, so we start at **2000**. We treat **2021 as partial** and keep it out of training and testing.

In [2]:
counts = df.groupby('release_year').size()
yearly = counts.loc[2000:2021].rename('titles').to_frame()
yearly['status'] = np.where(yearly.index==2021, 'partial', 'complete')
print('Latest date_added:', df['date_added'].max())
yearly.tail(8)

Latest date_added: 2021-09-25


,titles,status
release_year,,
2014,352,complete
2015,555,complete
2016,901,complete
2017,1030,complete
2018,1144,complete
2019,1029,complete
2020,953,complete
2021,592,partial


## Step 2: Analyze the yearly trend

In [3]:
full = yearly[yearly.status=='complete']['titles']
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(full.index, full.values, marker='o', label='Complete years')
ax.scatter([2021],[yearly.loc[2021,'titles']], color='red', zorder=3, label='2021 (partial, data ends Sep 25)')
ax.set_title('Titles in the catalog by release year'); ax.set_xlabel('Release year'); ax.set_ylabel('Number of titles')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('charts/01_titles_by_release_year.png', dpi=130); plt.show()
growth = full.pct_change()*100
print(growth.loc[2014:].round(1).to_string())

release_year
2014    23.1
2015    57.7
2016    62.3
2017    14.3
2018    11.1
2019   -10.1
2020    -7.4


In [4]:
by_type = df[df.release_year.between(2000,2020)].groupby(['release_year','type']).size().unstack(fill_value=0)
ax = by_type.plot(figsize=(10,5), marker='o')
ax.set_title('Movies vs TV shows by release year'); ax.set_ylabel('Number of titles'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('charts/02_movies_vs_tv_by_year.png', dpi=130); plt.show()

**What the trend shows:** release-year counts grow slowly until 2014 (352 titles), then climb steeply to a peak of 1,144 in 2018, and fall after that to 1,029 in 2019 and 953 in 2020. Split by type, the two lines tell different stories: movie counts fall after 2018 while TV show counts keep rising, so the catalog is tilting from films toward series. The 592 titles for 2021 are partial (the data ends 25 September 2021) and are excluded from training and testing.

## Step 3: Build forecasting models

We keep it simple, using only numpy so it runs anywhere. To test a model fairly we **hold out the last 4 complete years (2017 to 2020)**. Models only see data up to 2016, then predict 2017 to 2020 and we compare with what really happened. We never shuffle, because time order matters.

Models:
- **Naive:** every future year equals the last training year.
- **Average of last 3 years.**
- **Linear trend:** a straight line through the years.
- **Exponential trend:** a straight line on log(count), meaning constant percentage growth.
- **Holt (double exponential smoothing):** level plus trend that adapt over time.

Error measures: **MAE** (average miss, in titles) and **MAPE** (average miss, in %).

In [5]:
train = full.loc[2000:2016]; test = full.loc[2017:2020]
h = len(test)

def f_naive(y, h):  return np.repeat(y.iloc[-1], h).astype(float)
def f_avg3(y, h):   return np.repeat(y.iloc[-3:].mean(), h)
def f_linear(y, h):
    b, a = np.polyfit(y.index.values, y.values, 1)
    x = np.arange(y.index[-1]+1, y.index[-1]+1+h); return a + b*x
def f_exp(y, h):
    b, a = np.polyfit(y.index.values, np.log(y.values), 1)
    x = np.arange(y.index[-1]+1, y.index[-1]+1+h); return np.exp(a + b*x)
def f_holt(y, h, alpha=0.8, beta=0.3):
    level, trend = y.iloc[0], y.iloc[1]-y.iloc[0]
    for v in y.iloc[1:]:
        prev = level
        level = alpha*v + (1-alpha)*(level+trend)
        trend = beta*(level-prev) + (1-beta)*trend
    return np.array([level + (i+1)*trend for i in range(h)])

models = {'Naive':f_naive, 'Average of last 3':f_avg3, 'Linear trend':f_linear, 'Exponential trend':f_exp, 'Holt':f_holt}
preds = {n: f(train, h) for n, f in models.items()}
rows = []
for n, p in preds.items():
    err = test.values - p
    rows.append({'model':n, 'MAE':np.abs(err).mean(), 'MAPE %':(np.abs(err)/test.values).mean()*100})
scores = pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True).round(1)
scores

,model,MAE,MAPE %
0,Naive,138.0,12.9
1,Holt,204.0,20.8
2,Exponential trend,241.2,23.0
3,Average of last 3,436.3,41.8
4,Linear trend,454.4,43.4


In [6]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(train.index, train.values, color='gray', label='Train (2000-2016)')
ax.plot(test.index, test.values, color='black', marker='o', linewidth=2.5, label='Actual (holdout)')
for n, p in preds.items(): ax.plot(test.index, p, marker='x', linestyle='--', label=n)
ax.set_title('Holdout test: predictions vs what really happened'); ax.set_xlabel('Release year'); ax.set_ylabel('Number of titles')
ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('charts/03_holdout_comparison.png', dpi=130); plt.show()
ax = scores.set_index('model')['MAE'].plot.barh(figsize=(8,4), color='steelblue')
ax.set_xlabel('MAE (titles per year, lower is better)'); ax.set_title('Holdout error by model')
plt.tight_layout(); plt.savefig('charts/04_model_errors.png', dpi=130); plt.show()

**Reading the test:** the naive "same as last year" model wins with an MAE of 138 titles (12.9% MAPE). Holt (204) and the exponential trend (241) keep climbing after the 2018 peak and overshoot, while the 3-year average (436) and linear trend (454) were trained mostly on much lower years and undershoot. A simple model wins here because the series turns: after the 2018 peak the best guess for next year is close to last year, not a continuation of the old trend.

**Important caveat:** the holdout has only 4 points and contains a turning point (growth, then decline). Treat the ranking as a rough guide, not proof.

## Step 4: Visualize future predictions

We refit on all complete years (2000 to 2020) and forecast 2021 to 2025. For 2021 we also show the partial count actually in the data. A forecast for 2021 is a full-year number, so it will be higher than the partial count by design.

We show the best holdout model (by MAE) together with two simple alternatives so you can see the spread. I present the naive model because it had the lowest holdout error, and I show the 3-year average and the linear trend alongside it so the disagreement between models stays visible instead of hidden.

In [7]:
best = scores.loc[0,'model']; print('Best on holdout:', best)
H = 5; future = np.arange(2021, 2021+H)
final = {n: models[n](full, H) for n in [best, 'Naive', 'Average of last 3', 'Linear trend'] if n in models}
fc = pd.DataFrame(final, index=future).round(0).astype(int)
fc['Actual so far (partial)'] = [yearly.loc[2021,'titles']] + [np.nan]*(H-1)
fc

Best on holdout: Naive


,Naive,Average of last 3,Linear trend,Actual so far (partial)
2021,953,1042,977,592.0
2022,953,1042,1032,NaN
2023,953,1042,1088,NaN
2024,953,1042,1144,NaN
2025,953,1042,1199,NaN


In [8]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(full.index, full.values, color='black', marker='o', label='History (complete years)')
ax.scatter([2021],[yearly.loc[2021,'titles']], color='red', zorder=3, label='2021 so far (partial)')
for n in final: ax.plot([2020]+list(future), [full.iloc[-1]]+list(final[n]), linestyle='--', marker='x', label=f'Forecast: {n}')
ax.axvline(2020.5, color='gray', linestyle=':')
ax.set_title('Forecast of catalog titles by release year, 2021-2025 (historical sample, not Netflix plans)')
ax.set_xlabel('Release year'); ax.set_ylabel('Number of titles'); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('charts/05_forecast.png', dpi=130); plt.show()

## Step 5: Interpret results

- Release-year counts in this catalog rise slowly to 2014, jump from 2015 to a peak of 1,144 in 2018, then decline to 1,029 in 2019 and 953 in 2020. Movies drive the decline while TV shows keep rising.
- The naive model worked best on the 2017-2020 holdout: MAE 138 titles (12.9% MAPE), ahead of Holt (204), the exponential trend (241), the 3-year average (436) and the linear trend (454).
- For 2021-2025 the models disagree by hundreds of titles per year: the naive model holds flat at 953, the 3-year average at 1,042, and the linear trend climbs from 977 to 1,199. The honest reading is "roughly 950 to 1,200 catalog titles per release year", with wide uncertainty.

### Limits
1. **Partial 2021:** the data ends Sep 25, 2021, so 2021 is not a full year and was left out of training and testing.
2. **This is a catalog sample.** A title appears here only if it was on Netflix when the data was collected, and older or removed titles are missing. Falling counts after 2018 may partly reflect that recent titles had less time to be added, not that less content was made.
3. **Few data points.** About 20 yearly values and a 4-year test. Forecasts five years out are very uncertain.
4. **Not a business forecast.** It does not predict Netflix commissioning, budgets or viewer demand.

### Next steps (optional)
Forecast movies and TV shows separately, since their trends clearly differ; add prediction intervals to show the uncertainty; and try monthly counts for more data points.

**Dataset citation:** "Netflix Movies and TV Shows" by Shivam Bansal on Kaggle (https://www.kaggle.com/datasets/shivamb/netflix-shows), a snapshot of titles listed on Netflix as of September 2021.